In [184]:
import pandas as pd
import numpy as np
import json
import altair as alt

# Using ecostyles package (installed in environment)
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name='article')

In [185]:
hotdays = pd.read_excel("urban_cooling_Article_data.xlsx", sheet_name="Hotdays_map_data", header=0)

In [186]:
hotdays

,Year,Country,LA_name,LA_code,Number_of_hotdays
0,2024,England,West Devon,E07000047,0.117647
1,2024,Scotland,Na h-Eileanan Siar,S12000013,0.000000
2,2024,Scotland,Orkney Islands,S12000023,0.000000
3,2024,Scotland,Shetland Islands,S12000027,0.000000
4,2024,Wales,Isle of Anglesey,W06000001,0.000000
...,...,...,...,...,...
344,2024,England,Camden,E09000007,11.666667
345,2024,England,Hounslow,E09000018,11.023256
346,2024,England,City of London,E09000001,11.333333
347,2024,England,Hillingdon,E09000017,11.250000


In [187]:
local_authorities = pd.read_excel("urban_cooling_Article_data.xlsx", sheet_name="Top5_LAs_AnnualValue_chart", header=0)

In [188]:
local_authorities

,LA_name,Wales,Scotland,England
0,Cardiff,1.090840,NaN,NaN
1,Newport,0.544569,NaN,NaN
2,Rhondda Cynon Taf,0.403084,NaN,NaN
3,Caerphilly,0.228059,NaN,NaN
4,Swansea,0.214261,NaN,NaN
5,Glasgow City,NaN,0.446015,NaN
6,City of Edinburgh,NaN,0.409343,NaN
7,Aberdeen City,NaN,0.226104,NaN
8,North Lanarkshire,NaN,0.186493,NaN
9,West Lothian,NaN,0.154012,NaN


In [189]:
# Load Top5 LAs sheet and pivot Wales/Scotland/England to long format
import pandas as pd

local_authorities = pd.read_excel("urban_cooling_Article_data.xlsx", sheet_name="Top5_LAs_AnnualValue_chart", header=0)

# detect LA name column
la_candidates = ['LA_name','LA name','Local Authority','Local_Authority','LA']
la_col = next((c for c in local_authorities.columns if c in la_candidates), None)
if la_col is None:
    cols_lower = {c.lower(): c for c in local_authorities.columns}
    for cand in la_candidates:
        if cand.lower() in cols_lower:
            la_col = cols_lower[cand.lower()]
            break
if la_col is None:
    la_col = local_authorities.columns[0]

# detect region columns (Wales/Scotland/England)
region_names = ['Wales', 'Scotland', 'England']
region_cols = [c for c in local_authorities.columns if c.strip() in region_names]
if not region_cols:
    region_cols = [c for c in local_authorities.columns if c != la_col]

# melt to long format
local_authorities_long = local_authorities.melt(id_vars=[la_col], value_vars=region_cols, var_name='region', value_name='value')
local_authorities_long = local_authorities_long.rename(columns={la_col: 'LA_name'})[['region', 'LA_name', 'value']]
local_authorities_long = local_authorities_long.dropna(subset=['value'])

# save and show preview
out_path = 'Top5_LAs_AnnualValue_chart_long.csv'
local_authorities_long.to_csv(out_path, index=False)
print(f"Saved long-format CSV to {out_path}")
local_authorities_long


Saved long-format CSV to Top5_LAs_AnnualValue_chart_long.csv


,region,LA_name,value
0,Wales,Cardiff,1.090840
1,Wales,Newport,0.544569
2,Wales,Rhondda Cynon Taf,0.403084
3,Wales,Caerphilly,0.228059
4,Wales,Swansea,0.214261
20,Scotland,Glasgow City,0.446015
21,Scotland,City of Edinburgh,0.409343
22,Scotland,Aberdeen City,0.226104
23,Scotland,North Lanarkshire,0.186493
24,Scotland,West Lothian,0.154012


In [190]:
LAmap = "Local_Authority_Districts_December_2024_Boundaries_UK_BFE_-2486543043588325914.json" #simplified with mapshaper
with open(LAmap) as f:
    json_data = json.load(f)

In [196]:
import math
import copy

df = pd.read_excel("urban_cooling_Article_data.xlsx", sheet_name="Hotdays_map_data")
df = df[df["Year"] == 2024]
 
LAmap = "Local_Authority_Districts_December_2024_Boundaries_UK_BFE_-2486543043588325914.json"
with open(LAmap) as f:
    geojson_data = json.load(f)
 
data = alt.Data(values=geojson_data["features"])
 
# --- Correct for lon/lat distortion ---
# A degree of longitude covers less real-world distance than a degree of
# latitude the further from the equator you are. Plotting raw lon/lat with
# `identity` (no projection math) makes the UK look stretched east-west.
# Scaling longitude by cos(mean_latitude) approximates a proper
# equirectangular projection and fixes the proportions.
# (mercator/transverseMercator would do this "properly", but their
# auto-fit rendering is broken for this dataset in this environment —
# see earlier debugging. This manual correction sidesteps that bug.)
mean_lat = 55.0  # roughly the center latitude of Great Britain
scale_factor = math.cos(math.radians(mean_lat))
 
def scale_coords(coords):
    if isinstance(coords[0], (int, float)):
        return [coords[0] * scale_factor, coords[1]]
    return [scale_coords(c) for c in coords]
 
geojson_corrected = copy.deepcopy(geojson_data)
for feat in geojson_corrected["features"]:
    feat["geometry"]["coordinates"] = scale_coords(feat["geometry"]["coordinates"])
 
data = alt.Data(values=geojson_corrected["features"])
 
# --- Build the map ---
figure1_border = (
    alt.Chart(data)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        color=alt.Color(
            "Number_of_hotdays:Q",
            title="# of Hot Days",
            scale=alt.Scale(scheme="orangered"),
            legend=alt.Legend(
                            titleFontSize=10,
                            labelFontSize=9,
                            symbolSize=6,
                            orient="right",
                            direction="vertical",
                            offset=-50,
                        )
        ),
        tooltip=[
            alt.Tooltip("properties.LAD24NM:N", title="Local Authority"),
            alt.Tooltip("Number_of_hotdays:Q", title="# of Hot Days", format=".1f"),
        ],
    )
    .transform_lookup(
        lookup="properties.LAD24CD",
        from_=alt.LookupData(df, "LA_code", ["Number_of_hotdays"]),
    )
    .project(type="identity", reflectY=True)
    .properties(
        width=400,
        height=600,
        title=alt.TitleParams(
            text="Figure 1: Number of Hot Days in 2024 by Local Authority",
            subtitle=["Hot day is defined as the temperature exceeding 28°C.", "Temperatures are grid averages across area."],
            offset = 10
        ),

    )

)
  
figure1_transparent = (
    alt.Chart(data)
    .mark_geoshape(stroke="transparent", strokeWidth=0.5)
    .encode(
        color=alt.Color(
            "Number_of_hotdays:Q",
            title="# of Hot Days",
            scale=alt.Scale(scheme="orangered"),
            legend=alt.Legend(
                            titleFontSize=10,
                            labelFontSize=9,
                            symbolSize=6,
                            orient="right",
                            direction="vertical",
                            offset=-50,
                        )
        ),
        tooltip=[
            alt.Tooltip("properties.LAD24NM:N", title="Local Authority"),
            alt.Tooltip("Number_of_hotdays:Q", title="# of Hot Days", format=".1f"),
        ],
    )
    .transform_lookup(
        lookup="properties.LAD24CD",
        from_=alt.LookupData(df, "LA_code", ["Number_of_hotdays"]),
    )
    .project(type="identity", reflectY=True)
    .properties(
        width=400,
        height=600,
        title=alt.TitleParams(
            text="Figure 1: Number of Hot Days in 2024 by Local Authority",
            subtitle=["Hot day is defined as the temperature exceeding 28°C.", "Temperatures are grid averages across area."],
            offset = 10
        ),
    )
)
 


In [192]:
df = local_authorities_long
 
df_england = df[df["region"] == "England"].copy()
df_ws = df[df["region"].isin(["Wales", "Scotland"])].copy()

#fig 2
df_england = df_england.sort_values("value", ascending=False)
england_order = df_england["LA_name"].tolist()
 
fig2 = (
    alt.Chart(df_england)
    .mark_bar(color="#0063AF")
    .encode(
        y=alt.Y("LA_name:N", sort=england_order, title=None),
        x=alt.X("value:Q", title="Annual value (£ million, 2024 prices)"),
    )
    .properties(width=500, height=300, title="Figure 2: Top 5 England LAs by Annual Value in 2024")
)
 
fig2_text = (
    alt.Chart(df_england)
    .mark_text(align="left", dx=5, fontSize=13)
    .encode(
        y=alt.Y("LA_name:N", sort=england_order),
        x=alt.X("value:Q"),
        text=alt.Text("value:Q", format=".2f"),
    )
)
 
figure2 = (fig2 + fig2_text).configure_axis(
    labelFontSize=13, titleFontSize=13
).configure_title(fontSize=18)

In [193]:
figure2

alt.LayerChart(...)

In [194]:
#fig 3

wales_order = (
    df_ws[df_ws["region"] == "Wales"].sort_values("value", ascending=False)["LA_name"].tolist()
)
scotland_order = (
    df_ws[df_ws["region"] == "Scotland"].sort_values("value", ascending=False)["LA_name"].tolist()
)
ws_order = scotland_order + wales_order  # top-to-bottom order as in the chart
 
fig3 = (
    alt.Chart(df_ws)
    .mark_bar()
    .encode(
        y=alt.Y("LA_name:N", sort=ws_order, title=None),
        x=alt.X("value:Q", title="Annual value (£ million, 2024 prices)", axis=alt.Axis(tickCount=6)),
        color=alt.Color(
            "region:N",
            scale=alt.Scale(domain=["Scotland", "Wales"], range=[ "#179FDB", "#E6224B"])
        ),
    )
    .properties(width=500, height=300, title="Figure 3: Top 5 Scotland and Welsh LAs by Annual Value in 2024")
)
 
fig3_text = (
    alt.Chart(df_ws)
    .mark_text(align="left", dx=5, fontSize=13)
    .encode(
        y=alt.Y("LA_name:N", sort=ws_order),
        x=alt.X("value:Q", title="Annual value (£ million, 2024 prices)"),
        text=alt.Text("value:Q", format=".2f"),
    )
)
 
figure3 = (fig3 + fig3_text).configure_axis(
    labelFontSize=13, titleFontSize=13
).configure_title(fontSize=18)
 
figure3

alt.LayerChart(...)

In [197]:
styles.save(figure1_border, 'charts', 'fig1_border', width=300, height=500)
styles.save(figure1_transparent, 'charts', 'fig1_transparent', width=300, height=500)
styles.save(figure2, 'charts', 'fig2', width=500, height=300)
styles.save(figure3, 'charts', 'fig3', width=500, height=300)

**old version below**

In [ ]:

alt.data_transformers.disable_max_rows()
 
# --- Load data ---
df = pd.read_excel("urban_cooling_Article_data.xlsx", sheet_name="Hotdays_map_data")
df = df[df["Year"] == 2024]
 
LAmap = "Local_Authority_Districts_December_2024_Boundaries_UK_BFE_-2486543043588325914.json"
with open(LAmap) as f:
    geojson_data = json.load(f)
 
data = alt.Data(values=geojson_data["features"])
 
# --- Build the map ---
figure = (
    alt.Chart(data)
    .mark_geoshape(stroke="#ffffff", strokeWidth=0.7)
    .encode(
        color=alt.Color(
            "Number_of_hotdays:Q",
            title="Number Hot Days",
            scale=alt.Scale(scheme="orangered"),
            legend=alt.Legend(
                titleFontSize=10,
                labelFontSize=9,
                symbolSize=6,
                orient="right",
                direction="vertical",
                offset=10,
                titlePadding=4,
                labelPadding=4,
                legendY=20
            )
        ),
        tooltip=[
            alt.Tooltip("properties.LAD24NM:N", title="Local Authority"),
            alt.Tooltip("Number_of_hotdays:Q", title="Average Hot Days", format=".1f"),
        ],
    )
    .transform_lookup(
        lookup="properties.LAD24CD",
        from_=alt.LookupData(df, "LA_code", ["Number_of_hotdays"]),
    )
    # KEY FIX: mercator/equirectangular auto-fit was failing on this dataset
    # (rendering as one solid rectangle). identity + reflectY plots the
    # lon/lat coordinates directly, flipped to correct orientation, and
    # renders correctly for a country-sized area like the UK.
    .project(type="identity", reflectY=True)
    .properties(
        width=300,
        height=500,
        title=alt.TitleParams(
                    text="Figure 1: Number of Hot Days in 2024 by Local Authority",
                    subtitle=["Hot day is defined as the temperature exceeding 28°C"]
            )
    )
)
 
figure